# LineSight AI — обучение YOLO (Colab / Kaggle)
Запускайте ячейки по порядку. В Colab включите GPU: *Среда выполнения → Сменить тип → T4 GPU*.
Датасет: [sxiong/Power-equipment-image-dataset](https://huggingface.co/datasets/sxiong/Power-equipment-image-dataset) (лицензия MIT).
Для начала берём архив **transmission line object detection.zip** (530 МБ): объекты и дефекты ЛЭП с рамками.

In [ ]:
!git clone -b asyl_experiment https://github.com/asylbarakov/LineSight-AI.git
%cd LineSight-AI
!pip install -q ultralytics huggingface_hub

In [ ]:
!python -m ml.check_gpu

In [ ]:
from huggingface_hub import hf_hub_download
zip_path = hf_hub_download(
    repo_id="sxiong/Power-equipment-image-dataset", repo_type="dataset",
    filename="transmission line object detection.zip", local_dir="data/raw")
zip_path

## Подготовка датасета
Формат разметки (VOC / COCO / YOLO) определяется автоматически. Редкие классы (меньше 20 примеров) отбрасываются.
**Пришлите вывод этой ячейки** — по нему подберём классы для демо.

In [ ]:
!python -m ml.prepare_dataset --src "{zip_path}" --out data/datasets/lines_v0 --min-instances 20
!cat data/datasets/lines_v0/classes.json

## Обучение

In [ ]:
!python -m ml.train --data data/datasets/lines_v0/data.yaml --model yolov8s.pt --epochs 40 --imgsz 640 --batch -1

## Результат
`data/weights/best.pt` и `data/weights/metrics.json`. Скачайте `best.pt` и положите в `data/weights/` на машине с проектом.

In [ ]:
from google.colab import files  # в Kaggle: файл лежит в Output
files.download("data/weights/best.pt")
print(open("data/weights/metrics.json").read())

## Проверка на видео (по желанию)
`!python -m ml.check_detector data/weights/best.pt <video.mp4>`